# Noise models

反復測定から観測分散を推定する `ReplicateNoiseSingleTaskGP` を使い、latent function の不確かさと測定ノイズを分離します。


## 反復測定データ


In [ ]:
import torch
from robotorchan.models import ReplicateNoiseSingleTaskGP

torch.manual_seed(0)
torch.set_default_dtype(torch.double)
base_X = torch.tensor([[0.0], [0.5], [1.0]])
X = base_X.repeat_interleave(3, dim=0)
Y = torch.sin(2 * torch.pi * X) + 0.05 * torch.randn_like(X)


## 反復を集約してGPを構築


In [ ]:
model = ReplicateNoiseSingleTaskGP.from_replicates(X, Y)
model.fit()
print(model.raw_train_X)
print(model.raw_train_Yvar)
print(model.replicate_counts)


## Posterior


In [ ]:
test_X = torch.linspace(0, 1, 5).unsqueeze(-1)
posterior = model.posterior(test_X)
print(posterior.mean.squeeze(-1))


## Replicate-aware candidate proposal

反復観測から推定したノイズ分散を持つモデルで、予測平均最大の候補を選択します。


In [ ]:
candidate_grid = torch.linspace(0, 1, 101).unsqueeze(-1)
model.eval()
with torch.no_grad():
    scores = model.posterior(candidate_grid).mean.squeeze(-1)
candidate = candidate_grid[scores.argmax().item():scores.argmax().item() + 1]
candidate_Y = torch.sin(2 * torch.pi * candidate)
assert candidate.shape == (1, 1)
print("candidate:", candidate, "evaluated objective:", candidate_Y)


## Mixed replicate-aware candidate proposal

連続値とカテゴリ値の組合せごとに反復測定を集約し、元の混合入力空間で予測平均最大の候補を選びます。これは獲得関数の最適化ではありません。


In [ ]:
from robotorchan.models import MixedReplicateNoiseSingleTaskGP

mixed_base_X = torch.tensor(
    [[0.0, 0.0], [0.5, 0.0], [1.0, 0.0],
     [0.0, 1.0], [0.5, 1.0], [1.0, 1.0]]
)
mixed_replicate_X = mixed_base_X.repeat_interleave(3, dim=0)
mixed_replicate_Y = (
    torch.sin(2 * torch.pi * mixed_replicate_X[:, :1])
    + 0.2 * mixed_replicate_X[:, 1:2]
    + 0.03 * torch.randn(mixed_replicate_X.shape[0], 1)
)
mixed_model = MixedReplicateNoiseSingleTaskGP.from_replicates(
    mixed_replicate_X, mixed_replicate_Y, cat_dims=[1]
)
mixed_model.fit()
mixed_pool = torch.cartesian_prod(
    torch.linspace(0, 1, 21), torch.tensor([0.0, 1.0])
)
mixed_model.eval()
with torch.no_grad():
    mixed_scores = mixed_model.posterior(mixed_pool).mean.squeeze(-1)
mixed_candidate = mixed_pool[mixed_scores.argmax()].unsqueeze(0)
mixed_observation = (
    torch.sin(2 * torch.pi * mixed_candidate[:, :1])
    + 0.2 * mixed_candidate[:, 1:2]
)
assert mixed_candidate.shape == (1, 2)
assert mixed_candidate[0, 1].item() in (0.0, 1.0)
print("mixed candidate:", mixed_candidate)
print("evaluated objective:", mixed_observation)


## Input-dependent noise: single-task GPs

既存の `fit_heteroskedastic(iterations=1)` で平均モデルと対数ノイズモデルを交互学習します。平均のPosteriorと予測観測分散を分けて確認し、有限プールから候補を選びます。


In [ ]:
from robotorchan.models import (
    HeteroskedasticSingleTaskGP,
    MixedHeteroskedasticSingleTaskGP,
)

hetero_X = torch.linspace(0.05, 0.95, 10).unsqueeze(-1)
hetero_Y = (
    torch.sin(2 * torch.pi * hetero_X)
    + 0.03 * (1 + hetero_X) * torch.randn_like(hetero_X)
)
hetero_model = HeteroskedasticSingleTaskGP(hetero_X, hetero_Y)
hetero_model.fit_heteroskedastic(iterations=1)
hetero_model.eval()
hetero_pool = torch.linspace(0, 1, 31).unsqueeze(-1)
with torch.no_grad():
    hetero_scores = hetero_model.posterior(hetero_pool).mean.squeeze(-1)
    hetero_noise = hetero_model.predicted_noise(hetero_pool)
hetero_candidate = hetero_pool[hetero_scores.argmax()].unsqueeze(0)
assert torch.isfinite(hetero_noise).all()
print("heteroskedastic candidate:", hetero_candidate)
print("objective:", torch.sin(2 * torch.pi * hetero_candidate))

mixed_hetero_X = torch.cat(
    [hetero_X, torch.arange(10).remainder(2).to(hetero_X).unsqueeze(-1)],
    dim=-1,
)
mixed_hetero_Y = hetero_Y + 0.2 * mixed_hetero_X[:, 1:2]
mixed_hetero_model = MixedHeteroskedasticSingleTaskGP(
    mixed_hetero_X, mixed_hetero_Y, cat_dims=[1]
)
mixed_hetero_model.fit_heteroskedastic(iterations=1)
mixed_hetero_model.eval()
with torch.no_grad():
    mixed_hetero_scores = (
        mixed_hetero_model.posterior(mixed_pool).mean.squeeze(-1)
    )
    mixed_hetero_noise = mixed_hetero_model.predicted_noise(mixed_pool)
mixed_hetero_candidate = mixed_pool[mixed_hetero_scores.argmax()].unsqueeze(0)
assert torch.isfinite(mixed_hetero_noise).all()
print("mixed heteroskedastic candidate:", mixed_hetero_candidate)
print(
    "objective:",
    torch.sin(2 * torch.pi * mixed_hetero_candidate[:, :1])
    + 0.2 * mixed_hetero_candidate[:, 1:2],
)
